# Stage 0 — data explorer

Look a recording up by filename, list everything one speaker said, and actually *listen* to it.

Every non-audio field comes from the public ct2 dump (`load_index`, no gate).  Waveforms come
from gated `ivrit-ai/VoxKnesset` and need an accepted licence plus `HF_TOKEN` in the
environment; the first audio fetch also builds `stage2/shard_index.csv` (~12 min, once, then
cached).

**Disk guard.** Audio lands in `stage0/audio_cache/` (gitignored) under a hard `BUDGET_MB`
cap, least-recently-used files evicted first — the cache never grows past the budget, so
nothing here can fill the laptop.  `cache_status()` shows what is held, `clear_cache()` empties
it.

**Notebook-size guard.** `hear()` plays at most `MAX_PLAY_S` seconds, because every clip played
is embedded as base64 in the `.ipynb`.  Clear outputs before committing.

In [ ]:
import os, sys, time
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import Audio, display

HERE = Path.cwd() if Path.cwd().name == 'stage0' else Path.cwd() / 'stage0'
sys.path.insert(0, str(HERE.parent / 'stage2'))
from pipeline import load_index, materialize, read_wav, SR   # noqa: E402

CACHE       = HERE / 'audio_cache'    # gitignored
BUDGET_MB   = 500                     # hard cap on audio kept on disk
MAX_PLAY_S  = 60                      # hard cap on one playback (keeps the .ipynb small)
BYTES_PER_S = SR * 2                  # 16 kHz, 16-bit mono
CACHE.mkdir(exist_ok=True)
pd.set_option('display.max_colwidth', 120)

In [ ]:
try:
    IDX = load_index()                       # from the local snapshot
except Exception as e:
    print('local cache miss, downloading the public dump:', e)
    IDX = load_index(local_only=False)

DUR = IDX.set_index('filename').duration_s   # for size estimates
print(f'{len(IDX):,} recordings, {IDX.speaker_id.nunique()} speakers, '
      f'{IDX.duration_s.sum()/3600:,.0f} h')

## Cache — the disk guard

`fetch()` is the only thing here that writes audio.  Before downloading it estimates the size
from `duration_s` (16 kHz 16-bit mono = 32 kB/s ≈ 1.9 MB/min), evicts least-recently-used files
to make room, and refuses outright anything that cannot fit in `BUDGET_MB`.

In [ ]:
def _files():
    # cached wavs, least-recently-used first
    return sorted(CACHE.glob('*.wav'), key=lambda p: p.stat().st_atime)

def _used_mb():
    return sum(p.stat().st_size for p in CACHE.glob('*.wav')) / 2**20

def est_mb(fns):
    # download size of these filenames, in MB
    return float(DUR.reindex(pd.Index(fns)).fillna(0).sum()) * BYTES_PER_S / 2**20

def cache_status():
    fs = _files()
    print(f'{len(fs)} file(s), {_used_mb():.1f} MB of {BUDGET_MB} MB budget')
    return pd.DataFrame({'filename':  [p.name for p in fs],
                         'MB':        [round(p.stat().st_size / 2**20, 1) for p in fs],
                         'last_used': [time.strftime('%H:%M:%S', time.localtime(p.stat().st_atime))
                                       for p in fs]})

def clear_cache(keep=()):
    keep, n = set(keep), 0
    for p in _files():
        if p.name not in keep:
            p.unlink(); n += 1
    print(f'deleted {n} file(s), {_used_mb():.1f} MB left')

def _evict(need_mb):
    for p in _files():                       # oldest use first
        if _used_mb() + need_mb <= BUDGET_MB:
            return
        print(f'  evicting {p.name} ({p.stat().st_size / 2**20:.1f} MB)')
        p.unlink()

def fetch(fns):
    # Ensure these wavs are on disk, within budget.  Returns their paths.
    fns  = [fns] if isinstance(fns, str) else list(fns)
    bad  = [f for f in fns if f not in DUR.index]
    if bad:
        raise KeyError(f'not in the index: {bad[:5]}')
    todo = [f for f in fns if not (CACHE / f).exists()]
    if todo:
        mb = est_mb(todo)
        if mb > BUDGET_MB:
            raise MemoryError(f'{len(todo)} file(s) = {mb:.0f} MB > BUDGET_MB={BUDGET_MB}. '
                              f'Take fewer files, or raise BUDGET_MB deliberately.')
        print(f'downloading {len(todo)} file(s), ~{mb:.1f} MB')
        _evict(mb)
        materialize(todo, str(CACHE))
    for f in fns:
        os.utime(CACHE / f, None)            # mark as just used, for LRU
    return [CACHE / f for f in fns]

## 1. A recording, by filename

Filenames are `speaker_session_start_end.wav`.  `row(...)` shows every field the dump holds and
downloads nothing.

In [ ]:
WIDE = ['reference_text', 'model_transcription', 'segments_json']

def row(fn, full_text=False):
    # One recording's metadata.  Prints it readably, returns the Series.
    hit = IDX[IDX.filename == fn]
    if hit.empty:
        raise KeyError(f'{fn} is not in the index')
    r = hit.iloc[0]
    for k, v in r.items():
        if k not in WIDE:
            print(f'{k:>30} : {v}')
    print(f'{"cached":>30} : {(CACHE / fn).exists()}   (~{est_mb([fn]):.1f} MB)')
    for k in ('reference_text', 'model_transcription'):
        t = str(r[k])
        print(f'\n--- {k} ---\n{t if full_text else t[:600] + ("..." if len(t) > 600 else "")}')
    return r

In [ ]:
FILES = [
    '4416_66582_456_529.wav',      # <-- put your wav ids here
]

r = row(FILES[0])

## 2. Everything one speaker said

How many files, how long, over how many sessions — and what it would cost to keep on disk.

In [ ]:
def speaker(spk, sort='duration'):
    # All recordings of one speaker + a summary.  Returns the DataFrame.
    g = IDX[IDX.speaker_id == int(spk)]
    if g.empty:
        raise KeyError(f'no speaker {spk}; try sorted(IDX.speaker_id.unique())')
    g = g.sort_values('filename' if sort == 'time' else 'duration_s',
                      ascending=(sort == 'time'))
    d = g.iloc[0]
    print(f'speaker {spk}  |  gender {d.gender}  age {g.age.min():.0f}-{g.age.max():.0f}  '
          f'{d.speaker_place_of_birth}')
    print(f'files     : {len(g):,}')
    print(f'sessions  : {g.session.nunique()}')
    print(f'audio     : {g.duration_s.sum()/3600:.2f} h  '
          f'(median file {g.duration_s.median():.0f} s, longest {g.duration_s.max():.0f} s)')
    print(f'all of it : ~{est_mb(g.filename):,.0f} MB on disk  (budget is {BUDGET_MB} MB)')
    by_sess = g.groupby('session').agg(files=('filename', 'size'),
                                       minutes=('duration_s', lambda s: round(s.sum() / 60, 1)))
    print(f'\nlongest {min(10, len(by_sess))} of {len(by_sess)} sessions:')
    print(by_sess.sort_values('minutes', ascending=False).head(10).to_string())
    return g[['filename', 'session', 'duration_s', 'split', 'reference_text']]

SPEAKER = 4416                       # <-- put your speaker id here
files = speaker(SPEAKER)
files.head(20)

## 3. Listen

`hear(fn)` downloads the file if it is not cached and plays its first `MAX_PLAY_S` seconds.
`hear(fn, 120, 150)` plays 120 s → 150 s.  The whole file stays on disk; only the requested
window is embedded in the notebook.

In [ ]:
def hear(fn, start=0.0, end=None, text=True):
    # Play a window of one recording.  Downloads it first, within budget.
    p   = fetch(fn)[0]
    dur = float(DUR.get(fn, np.nan))
    end = min(dur if end is None else end, start + MAX_PLAY_S)
    x   = read_wav(str(p), start, end)
    print(f'{fn}   {start:.0f}-{start + len(x)/SR:.0f}s of {dur:.0f}s'
          + ('' if end >= dur else f'   (capped at MAX_PLAY_S={MAX_PLAY_S}s)'))
    if text:
        print('\n' + str(IDX.loc[IDX.filename == fn, 'reference_text'].iloc[0])[:400])
    display(Audio(x, rate=SR))

hear(FILES[0])

In [ ]:
# a few of one speaker's files, shortest first -- cheap to fetch, quick to skim
for fn in files.sort_values('duration_s').filename.head(3):
    hear(fn, text=False)

## 4. Housekeeping

In [ ]:
cache_status()

In [ ]:
# clear_cache()                       # delete every cached wav
# clear_cache(keep=FILES)             # delete all but these